# Used car price tier classification
**Dataset:** CarDekho Used Car Dataset (Extended, v3)  
**Source:** https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho?select=Car+details+v3.csv

***
# 0. Dataset Description
## Source
The dataset is the **CarDekho Used Car Dataset (v3)** from Kaggle:  
https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho

The "Car details v3.csv" file is downloaded from my public github repository.

## Problem Context
CarDekho is a Indian used-car marketplace. This dataset contains listings of second-hand cars with various tehnical and commercial attributes. The classification target is the price tier of a car (budget, mid-range, premium) derived by binning the `selling_price` column into 3 categories.

Real-world relevance: a buyer, seller, or dealership could use such a classifier to automatically assess which market segment a car belongs to based on its specifications - witout knowing its price.

## Feature Description
| # | Feature | Meaning | Type | Examples |
|---|---|---|---|---|
| 1 | `year` | Year the car was manufactured | Numerical (discrete) | 2005, 2010, 2020 |
| 2 | `km_driven` | Total km of the car | Numerical (continuous) | 50000, 200000 |
| 3 | `fuel` | Type of fuel used | Nominal Categorical | Petrol, Diesel, CNG, LPG, Electric |
| 4 | `seller_type` | Who is selling the car | Nominal Categorical | Individual, Dealer, Trustmark Dealer |
| 5 | `transmission` | Gearbox type | Binary Categorical | Manual, Automatic |
| 6 | `owner` | Number of previous owners | Ordinal Categorical | First owner, Second Owner, ... |
| 7 | `mileage` | Fuel efficency(km/l or km/kg) | Numerical (continuous) | 23.4, 21.14 |
| 8 | `engine` | How big the engine is in CC | Numerical (contnuous) | 1248, 1498, 796 |
| 9 | `max_power` | Maximum engine power in bhp | Numerical (continuous) | 74, 103.52, 90 |
| 10 | `seats` | Number of seats | Numerical (discrete) | 4,5,7 |
| - | `price_tier`(target) | Price category | Nominal Categorical | budget, mid, premium | 

[`name` is dropped (too many unique strings). `torque` is dropped (inconsistent units). `selling_price` is dropped (target variable is derived from it).]

## Dataset Size and Class Distribution
- **Instances:** 8.128
- **Input Features:** 10
- **Target classes:** 3 (budget / mid / premium) 
- Class distribution is balanced by construction (~33% per class) 





***
## Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings,os,re
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, accuracy_score
)

RANDOM_STATE = 42
print('All imports successful.')

***
# 1. Data Loading and Preprocessing

## 1.1 Download and Load Dataset

In [ ]:
url = "https://raw.githubusercontent.com/Raulqy8/Classification-Task-UsedCarPriceTierClassification/main/Car%20details%20v3.csv"

df_raw = pd.read_csv(url)
print(f'Raw shape: {df_raw.shape}')
df_raw.head()

## 1.2 Overview

In [ ]:
print('Columns: ', df_raw.columns.tolist())
print('\nData types:')
print(df_raw.dtypes)
print('\nMissing values per column:')
print(df_raw.isnull().sum())
print('Basic sattistics:')
df_raw.describe(include='all')

## 1.3 Data Cleaning

The `mileage`, `engine` and `max_power` are stored as strings("19.7 kmpl"). We strip the units and convert them to float. Rows with unparseable values become NaN and will be handled by imputation in the preprocessing stage.

In [ ]:
df = df_raw.copy()
df.drop(columns=['name', 'torque'], inplace=True, errors='ignore')

def extract_numeric(series):
    return pd.to_numeric(series.str.extract(r'(\d+\.?\d*)')[0], errors='coerce')

df['mileage'] = extract_numeric(df['mileage'])
df['engine'] = extract_numeric(df['engine'])
df['max_power'] = extract_numeric(df['max_power'])

print('Dtypes after conversion: ')
print(df.dtypes)
print('\nMissing values after conversion:')
print(df.isnull().sum())

## 1.4 Create Target Variable

We derive the classification target `price_tier` by splitting `selling_price` into 3 quantile-based bins:
- budget: bottom 33%
- mid: middle 33%
- premium: top 33%

Quantile-based binning is chosen because it produces balanced classes regardless of the price distribution. After creating the target, `selling_price` is dropped from the features.

In [ ]:
df['price_tier'] = pd.qcut(df['selling_price'], q=3, labels=['budget', 'mid', 'premium'])
df.drop(columns=['selling_price'], inplace=True)
df.dropna(subset=['price_tier'], inplace=True)

print(f'Dataset shape after traget creation: {df.shape}' )
print('\nClass distribution:')
print(df['price_tier'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
counts = df['price_tier'].value_counts()
counts.plot(kind='bar', ax=axes[0], color=['blue', 'green', 'red'], edgecolor='black')
axes[0].set_title('Price tier distribution')
axes[0].set_xlabel('Price Tier')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
axes[1].pie(counts, labels=counts.index, autopct='%1.1f%%', colors=['blue', 'green', 'red'])
axes[1].set_title('Price tier distribution (%)')
plt.tight_layout()
plt.savefig('price_tier_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

## 1.5 Encoding and Preprocessing Strategy
We have three groups of features requiring different strategies:

| **Group** | **Features** | **Strategy** | **Reason** |
|---|---|---|---|
| **Numerical** | `year`, `km_driven`, `mileage`, `engine`, `max_power`, `seats` | Median imputation; StandardScaler only for SVM | NaN values present; scale matters for SVM but not trees
| **Ordinal categorical** | `owner` | OrdinalEncoder with explicit order | "First Owner" < "Second Owner" | 
| **Nominal Categorical** | `fuel`, `seller_type`, `transmission` | OneHotEncoder | No natural order; integer encoding would imply false ordering |

We don't scale for tree-based models because Desicion Trees and Radnom Forests split on feature thresholds which means scaling has no effect on their output. Scaling is applied only inside the SVM pipeline to prevent data leaks.

In [ ]:
TARGET = 'price_tier'
feature_cols = [c for c in df.columns if c != TARGET]
X_raw = df[feature_cols].copy()

le= LabelEncoder()
y = le.fit_transform(df[TARGET])
print(f'Target classes: {le.classes_} -> encoded as {list(range(len(le.classes_)))}')

numerical_features = ['year', 'km_driven', 'mileage', 'engine', 'max_power', 'seats']
ordinal_features = ['owner']
nominal_features = ['fuel', 'seller_type', 'transmission']
owner_order = ['First Owner', 'Second Owner', 'Third Owner', 'Fourth & Above Owner', 'Test Drive Car']

print(f'\nNumerical ({len(numerical_features)}): {numerical_features}')
print(f'Ordinal ({len(ordinal_features)}): {ordinal_features}')
print(f'Nominal ({len(nominal_features)}): {nominal_features}')

In [ ]:
# Sub pipelines
numerical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

numerical_transformer_scaled = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

ordinal_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=[owner_order],
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

nominal_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numerical_transformer, numerical_features),
    ('ord', ordinal_transformer, ordinal_features),
    ('nom', nominal_transformer, nominal_features)
])

preprocessor_scaled = ColumnTransformer([
    ('num', numerical_transformer_scaled, numerical_features),
    ('ord', ordinal_transformer, ordinal_features),
    ('nom', nominal_transformer, nominal_features)
])

print('Preprocessing pipelines defined.')

In [ ]:
# Transformed output
X_preview = preprocessor.fit_transform(X_raw)
ohe_cols = preprocessor.named_transformers_['nom']['encoder'].get_feature_names_out(nominal_features)
all_feature_names = numerical_features + ordinal_features + list(ohe_cols)

print(f'Transformed matrix: {X_preview.shape}')
print(f'Total features: {len(all_feature_names)}')
print(f'Feature names: {all_feature_names}')

In [ ]:
# View numerical features
fig, axes = plt.subplots(2,3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(numerical_features):
    df[col].dropna().hist(bins=30, ax=axes[i], color='steelblue', edgecolor='black', alpha=0.8)
    axes[i].set_title(col)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')

plt.suptitle('Numerical feature distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('numerical_distributions.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# View categorical features
cat_cols = ordinal_features + nominal_features
fig,axes = plt.subplots(1,len(cat_cols), figsize=(16, 4))
for i, col in enumerate(cat_cols):
    df[col].value_counts().plot(kind='bar', ax=axes[i], color='lightblue', edgecolor='black')
    axes[i].set_title(col)
    axes[i].tick_params(axis='x', rotation=30)
    
plt.suptitle('Categorical feature distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('categorical_distributions.png', dpi=100, bbox_inches='tight')
plt.show()

***
# 2. Train/Test Split

We use an **80/20 split** with `stratify=y`.

We use **stratification** because it guarantees that each class appears in both splits at the correct proportion, making the evaluation reliable. (after row-dropping (NaN removal), any imbalance could result in the smaller class being underrepresented or absent in the test set)

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Training set: {X_train_raw.shape[0]} samples')
print(f'Test set: {X_test_raw.shape[0]} samples')

full_dist = pd.Series(y).value_counts(normalize=True).sort_index()
train_dist = pd.Series(y_train).value_counts(normalize=True).sort_index()
test_dist = pd.Series(y_test).value_counts(normalize=True).sort_index()

full_dist = full_dist.reindex([0, 1, 2], fill_value=0)
train_dist = train_dist.reindex([0, 1, 2], fill_value=0)
test_dist = test_dist.reindex([0, 1, 2], fill_value=0)

dist_df = pd.DataFrame(
    {'Full' : full_dist, 'Train' : train_dist, 'Test' : test_dist}
)
dist_df.index = le.classes_

print('\nClass proportions:')
print((dist_df*100).round(2).to_string()+' %')

dist_df.plot(kind='bar', figsize=(8,4), edgecolor='black')
plt.title('Class distribution: Full / Train / Test')
plt.ylabel('Proportion')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('stratification_check.png', dpi=100, bbox_inches='tight')
plt.show()

***
# 3. Performance Metric Selection

**Chosen primary metric: Weighted F1-score**  
F1 is the harmonic mean of precision and recall. The **weighted** variant averages per-class F1 scores weighted by their support, making it robust to any residual class imbalance. We additionally report the full **classification report** (per-class precision, recall, F1) and **confusion matrix** for each model

In [ ]:
LABLE_NAMES = le.classes_
results = []

def evaluate_model(name, pipeline, X_tr, y_tr, X_te, y_te):
    pipeline.fit(X_tr, y_tr)
    y_pred_tr = pipeline.predict(X_tr)
    y_pred_te = pipeline.predict(X_te)

    f1_tr = f1_score(y_tr, y_pred_tr, average='weighted')
    f1_te = f1_score(y_te, y_pred_te, average='weighted')
    acc_tr = accuracy_score(y_tr, y_pred_tr)
    acc_te = accuracy_score(y_te, y_pred_te)

    print(f'Model: {name}')
    print(f'Traing accuracy: {acc_tr:.4f} | Train Weighted F1: {f1_tr:.4f}')
    print(f'Test accuracy: {acc_te:.4f} | Test Weighted F1: {f1_te:.4f}')
    
    gap = f1_tr - f1_te
    if(gap > 0.07):
        print(f'Overffiting warning (train test F1 gap: {gap:.4f})')
    elif(f1_te < 0.70):
        print(f'Underfitting warning (test F1: {f1_te:.4f})')
    else:
        print(f'Good generalization (gap = {gap:.4f})')

    print(f'Classification report (Test set):')
    print(classification_report(y_te, y_pred_te, target_names=LABLE_NAMES))

    cm = confusion_matrix(y_te, y_pred_te)
    fig, ax = plt.subplots(figsize=(5,4))
    ConfusionMatrixDisplay(cm, display_labels=LABLE_NAMES).plot(ax=ax, cmap='Blues', colorbar=False)

    ax.set_title(f'Confusion Matrix - {name}')
    plt.tight_layout()
    safe = re.sub(r'[^\w]', '_', name)
    plt.savefig(f'cm_{safe}.png', dpi=100, bbox_inches='tight')
    plt.show()

    return {'model' : name, 'train_f1' : f1_tr, 'test_f1' : f1_te, 'train_acc' : acc_tr, 'test_acc' : acc_te}

print('Eval helper defined')

***
# 4. Feature Selection

We use **Mutual Information** to rank all features by how much information they share with the target variable. MI is a model-agnostic and works with both numerical and categorical (encoded) data.

We then compare a full model vs a slim model (top 6 features by MI) using 5-fold-cross-validation to decide whether dropping features is beneficial.

In [ ]:
# Fit preprocessor on training data only
X_train_proc = preprocessor.fit_transform(X_train_raw)
X_test_proc = preprocessor.transform(X_test_raw)

ohe_cols_fitted = preprocessor.named_transformers_['nom']['encoder'].get_feature_names_out(nominal_features)
all_feature_names = numerical_features + ordinal_features + list(ohe_cols_fitted)

mi_scores = mutual_info_classif(X_train_proc, y_train, random_state=RANDOM_STATE)
mi_df = pd.DataFrame({'feature' : all_feature_names, 'mi_score' : mi_scores})
mi_df = mi_df.sort_values('mi_score', ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(data=mi_df, x='mi_score', y='feature', palette='viridis')
plt.title('Feature importance - after encoding')
plt.tight_layout()
plt.savefig('mutual_info.png', dpi=100, bbox_inches='tight')
plt.show()
print(mi_df.to_string(index=False))

In [ ]:
# Compare full vs slim model with 5-fold CV
top_k = 6
top_features_names = mi_df.head(top_k)['feature'].tolist()
top_idx = [list(mi_df['feature']).index(f) for f in top_features_names]

X_train_slim = X_train_proc[:, top_idx]

dt_full = DecisionTreeClassifier(random_state=RANDOM_STATE)
dt_slim = DecisionTreeClassifier(random_state=RANDOM_STATE)

cv_full = cross_val_score(dt_full, X_train_proc, y_train, cv=5, scoring='f1_weighted')
cv_slim = cross_val_score(dt_slim, X_train_slim, y_train, cv=5, scoring='f1_weighted')

print(f'Full model ({X_train_proc.shape[1]} encoded features) - CV Weighted F1: {cv_full.mean():.4f} +- {cv_full.std():.4f}')
print(f'Slim model (top {top_k} features) - CV Weighted F1: {cv_slim.mean():.4f} +- {cv_slim.std():.4f}')
print(f'\nTop {top_k} features selected')
print(mi_df.head(top_k)[['feature', 'mi_score']].to_string(index=False))

**Conclusion on feature selection**  
The MI ranking shows that `max_power`, `engine` and `mileage` are typically the strongest predictors of price tier, while features like `seats` or `seller_type` contirbute less. Since there are only 10 original features, the performance gap is small between full and slim models, so **we keep all features** in the final models to avoid unnecessary information loss.

***
# 5. Model Building, Evaluation and Optimisation

We build three models:
1. **Decision Tree** - interpretable baseline, prone to overfitting
2. **SVM** - kernel based classifier requiring scaled features
3. **Random Forest** - typically best generalisation

Each model is wrapped in a complete sklearn `Pipeline`. For each: baseline -> overfitting analysis -> hyperparameter tunning -> final test evaluation

## 5.1 Decision Tree

In [ ]:
dt_baseline_pipe = Pipeline([
    ('prep', preprocessor),
    ('clf', DecisionTreeClassifier(random_state=RANDOM_STATE))
])
r = evaluate_model('Decision Tree - Baseline', dt_baseline_pipe, X_train_raw, y_train, X_test_raw, y_test)
results.append(r)

In [ ]:
# Overfitting / underfitting analysis

depths = range(1,20)
train_f1s, test_f1s = [], []
for d in depths:
    pipe = Pipeline([
        ('prep', preprocessor),
        ('clf', DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE))
    ])
    pipe.fit(X_train_raw, y_train)
    train_f1s.append(f1_score(y_train, pipe.predict(X_train_raw), average='weighted'))
    test_f1s.append(f1_score(y_test, pipe.predict(X_test_raw), average='weighted'))

plt.figure(figsize=(10,4))
plt.plot(depths, train_f1s,'o-', label='Train F1')
plt.plot(depths, test_f1s,'s-', label='Test F1')
plt.xlabel('Max Depth')
plt.ylabel('Weighted F1 Score')
plt.title('Decision Tree Depth vs Weighted F1 Score (Overfitting Analysis)')
plt.legend()
plt.tight_layout()
plt.savefig('dt_depth_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
dt_param_grid = {
    'clf__max_depth' : [None, 5, 8, 12, 16, 20],
    'clf__min_samples_split' : [2, 4, 8, 16],
    'clf__min_samples_leaf' : [1, 2, 4, 8],
    'clf__criterion' : ['gini', 'entropy'],
    'clf__max_features' : [None, 'sqrt', 'log2']
}

dt_pipe = Pipeline([
    ('prep', preprocessor),
    ('clf', DecisionTreeClassifier(random_state=RANDOM_STATE))
])

dt_grid = GridSearchCV(dt_pipe, dt_param_grid, cv=5, scoring='f1_weighted', n_jobs=-1, verbose=0)
dt_grid.fit(X_train_raw, y_train)

print(f'Best DT params: {dt_grid.best_params_}')
print(f'Best CV weighted F1: {dt_grid.best_score_:.4f}')


In [ ]:
r = evaluate_model('Decision Tree - Tuned', dt_grid.best_estimator_, X_train_raw, y_train, X_test_raw, y_test)
results.append(r)

dt_clf = dt_grid.best_estimator_.named_steps['clf']
fig, ax = plt.subplots(figsize=(22,8))
plot_tree(dt_clf, feature_names=all_feature_names, class_names=LABLE_NAMES, filled=True, rounded=True, max_depth=3, ax=ax, fontsize=8)
plt.title('Decision tree tuned - first 3 levels')
plt.tight_layout()
plt.savefig('dt_tree.png', dpi=100, bbox_inches='tight')
plt.show()

## 5.2 Support Vector Machine (SVM)
SVMs optimise a decision boundary that maximises the margin between classes. Because this margin is defined in terms of distances, SVMs need feature scaling - a feature with values in the thousands (`km_driven`) would dominate. We use preproessor_scaled (which includes `Standard Scaler`) inside the SVM pipeline. The scaler is only fitted on the training data.

In [ ]:
svm_baseline_pipe = Pipeline([
    ('prep', preprocessor_scaled),
    ('clf', SVC(random_state=RANDOM_STATE))
])
r = evaluate_model('SVM - Baseline', svm_baseline_pipe, X_train_raw, y_train, X_test_raw, y_test)
results.append(r)

In [ ]:
svm_param_grid = {
    'clf__C' : [0.1, 1, 10, 100],
    'clf__kernel' : ['rbf', 'poly', 'linear'],
    'clf__gamma' : ['scale', 'auto']
}
svm_pipe = Pipeline([
    ('prep', preprocessor_scaled),
    ('clf', SVC(random_state=RANDOM_STATE))
])

svm_grid = GridSearchCV(svm_pipe, svm_param_grid, cv=5, scoring='f1_weighted', n_jobs=-1, verbose=0)
svm_grid.fit(X_train_raw, y_train)

print(f'Best SVM params: {svm_grid.best_params_}')
print(f'Best CV weighted F1: {svm_grid.best_score_:.4f}')

In [ ]:
r = evaluate_model('SVM - Tuned', svm_grid.best_estimator_, X_train_raw, y_train, X_test_raw, y_test)
results.append(r)

## 5.3 Random Forest (Ensemble Method)
Random forest trains more decision trees each using a random subset of features at each split. The final prediction it the majority. This method reduces overfitting. We use `RandomizedSearchCV` because random sampling is efficient and typically finds near-optimal configurations. 

In [ ]:
rf_baseline_pipe = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(random_state=RANDOM_STATE))
])
r = evaluate_model('Random Forest - Baseline', rf_baseline_pipe, X_train_raw, y_train, X_test_raw, y_test)
results.append(r)

In [ ]:
rf_param_dist = {
    'clf__n_estimators' : [50, 100, 200, 300],
    'clf__max_depth' : [None, 5, 10, 15, 20],
    'clf__min_samples_split' : [2, 5, 10],
    'clf__min_samples_leaf' : [1, 2, 4],
    'clf__max_features' : ['sqrt', 'log2', None]
}

rf_pipe = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(random_state=RANDOM_STATE))
])

rf_random = RandomizedSearchCV(rf_pipe, rf_param_dist, n_iter=40, cv=5, scoring='f1_weighted', n_jobs=-1, random_state=RANDOM_STATE, verbose=0)
rf_random.fit(X_train_raw, y_train)

print(f'Best RF params: {rf_random.best_params_}')
print(f'Best CV weighted F1: {rf_random.best_score_:.4f}')

In [ ]:
r = evaluate_model('Random Forest - Tuned', rf_random.best_estimator_, X_train_raw, y_train, X_test_raw, y_test)
results.append(r)

rf_clf = rf_random.best_estimator_.named_steps['clf']
rf_imp_df = pd.DataFrame({'feature' : all_feature_names, 'importance' : rf_clf.feature_importances_})
rl_imp_df = rf_imp_df.sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(10,5))
sns.barplot(data=rl_imp_df, x='importance', y='feature', palette='magma')
plt.title('Random Forest - Top 15 Feature Importances')
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

## 5.4 Model Comparison Summary

In [ ]:
results_df = pd.DataFrame(results).sort_values('test_f1', ascending=False)
print('\n=== MODEL COMPARISON ===')
print(results_df[['model','train_acc','test_acc','train_f1','test_f1']].to_string(index=False))

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(results_df))
w = 0.35
bars1 = ax.bar(x - w/2, results_df['train_f1'], w, label='Train F1', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + w/2, results_df['test_f1'],  w, label='Test F1',  color='darkorange', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(results_df['model'], rotation=20, ha='right')
ax.set_ylabel('Weighted F1')
ax.set_title('Model Comparison — Train vs Test Weighted F1')
ax.legend()
ax.set_ylim(0, 1.07)
for b in list(bars1) + list(bars2):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
            f'{b.get_height():.3f}', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

***
# 6. New Features
| **New Feature** | **Formula** | **Rationale** |
|---|---|---|
|`car_age`| `2026 - year` | Age of car in years. More interpretable than raw year |
|`km_per_year`| `km_driven / (car_age+1)` | Anual usage intensit. A car used more heavily depricates faster regardless of total km |
|`power_to_engine`| `max_power / engine` | Power density (hp/CC). Premium / sprots cars tend to have higher values |

In [ ]:
def add_new_features(X_df):
    X_new = X_df.copy()
    X_new['car_age'] = 2026 - X_new['year']
    X_new['km_per_year'] = X_new['km_driven'] / (X_new['car_age'] + 1)
    X_new['power_to_engine'] = X_new['max_power'] / (X_new['engine'].replace(0, np.nan))
    return X_new

X_train_new = add_new_features(X_train_raw)
X_test_new = add_new_features(X_test_raw)

new_features = ['car_age', 'km_per_year', 'power_to_engine']
print(f'Features before: {X_train_raw.shape[1]}')
print(f'Features after: {X_train_new.shape[1]}')
print(f'New features added: {new_features}')

In [ ]:
# Rebuild preprocessors with new features

new_num_features = numerical_features + new_features

preprocessor_new = ColumnTransformer([
    ('num', numerical_transformer, new_num_features),
    ('ord', ordinal_transformer, ordinal_features),
    ('nom', nominal_transformer, nominal_features)
])

preprocessor_scaled_new = ColumnTransformer([
    ('num', numerical_transformer_scaled, new_num_features),
    ('ord', ordinal_transformer, ordinal_features),
    ('nom', nominal_transformer, nominal_features)
])

print('New preprocessors ready')

In [ ]:
X_train_new_proc = preprocessor_new.fit_transform(X_train_new)
ohe_new = preprocessor_new.named_transformers_['nom']['encoder'].get_feature_names_out(nominal_features)
all_cols_new = new_num_features + ordinal_features + list(ohe_new)

mi_new = mutual_info_classif(X_train_new_proc, y_train, random_state=RANDOM_STATE)
mi_new_df = pd.DataFrame({'feature' : all_cols_new, 'mi_score' : mi_new})
mi_new_df = mi_new_df.sort_values('mi_score', ascending=False)

colors = ['red' if f in new_features else 'steelblue' for f in mi_new_df['feature']]

plt.figure(figsize=(10,7))
sns.barplot(data=mi_new_df, x='mi_score', y='feature', palette=colors)
plt.title('Mutual info - all features ( red = new features )')
plt.tight_layout()
plt.savefig('mutual_info_all.png', dpi=100, bbox_inches='tight')
plt.show()
print(mi_new_df.head(12).to_string(index=False))

In [ ]:
# Re-evaluate best models with new features
best_dt_params = {k.replace('clf__', ''): v for k, v in dt_grid.best_params_.items()}
best_rd_params = {k.replace('clf__', ''): v for k, v in rf_random.best_params_.items()}

dt_new_pipe = Pipeline([
    ('prep', preprocessor_new),
    ('clf', DecisionTreeClassifier(**best_dt_params, random_state=RANDOM_STATE))
])
r_dt_new = evaluate_model('Decision Tree - Tuned with new features', dt_new_pipe, X_train_new, y_train, X_test_new, y_test)
results.append(r_dt_new)

rf_new_pipe = Pipeline([
    ('prep', preprocessor_new),
    ('clf', RandomForestClassifier(**best_rd_params, random_state=RANDOM_STATE))
])
r_rf_new = evaluate_model('Random Forest - Tuned with new features', rf_new_pipe, X_train_new, y_train, X_test_new, y_test)
results.append(r_rf_new)

In [ ]:
orig_dt = next(r for r in results if r['model'] == 'Decision Tree - Tuned')
orig_rf = next(r for r in results if r['model'] == 'Random Forest - Tuned')

print('New Features impact:')
print(f"Decision Tree: {orig_dt['test_f1']:.4f} -> {r_dt_new['test_f1']:.4f}"
      f"({'improved' if r_dt_new['test_f1'] > orig_dt['test_f1'] else 'no gain/worse'})")
print(f"Random Forest: {orig_rf['test_f1']:.4f} -> {r_rf_new['test_f1']:.4f}"
      f"({'improved' if r_rf_new['test_f1'] > orig_rf['test_f1'] else 'no gain/worse'})")

**New features analysis:**
- `car_age` directly represents depricaion - it ranks pretty high in the MI
- `km_per_year` normalises usage by age. A 2 year old car with 100,000 km is not the same as a 10 year old car with the same total km which is why it ranked higher than `km_driven`.
- `power_to_engine` (bhp/CC) measures engine efficency. Premium vehicles tend to have higher values.


 | Model | Original | With new features | Change |
 |---|---|---|---|
 |**Decision Tree** | 0.8405 | 0.8491 | +0.86% |
 |**Random Forest** | 0.8762 | 0.8713 | -0.49% |

 **Observations**:
 - **Decision Tree slightly improved**: Simple models benefit from more explicit features. The new features provide clearer decision boundaries.
 - **Random Forest declined slightly**: Ensemble methods with random feature selection already perform implicit feature engineering. Adding new features only introduces minor noise decreasing the model's accuracy.

 **Conclusion on adding these new features**: The modifications of the models are very marginal. Even though they worked as expected the performance increase is too small to justify adding this new features to the models.

***
## Final Summary

| **Phase** | **Decision** | **Justification** |
|---|---|---|
|Dataset|CarDekho Used Car v3|10 real-world features, 8000+ instances, realistic clasification task|
|Target|Price tier(budget / mid / premium)| Quantile binning of `selling_price` -> balanced classes|
|Encoding|Ordinal for `owner`, OHE for nominal, median imputation for NaNs| Preserves ordinal structure|
|Scaling|StandardScaler only for SVM| Trees are scale-invariant; SVM is not|
|Split|80/20 stratified| Ensures all classes represented proportionally in both subsets|
|Metric|Weighted F1-score|Balanced precision/recall|
|Feature selection| All 10 features retained | MI shows all contribute; slim model not significantly better|
|Models|Decision Tree + SVM + Random Forest| Covers interpretable, kernel-based and enseble methods|
|Tuning|GridSearchCV(DT, SVM) + RandomizedSearchCV(RF)|RF search space is too large for full grid|
|New features| `car_age`, `km_per_year`, `power_to_engine`| New features someone would be interested in| 

**Best model: Random Forrest (tunned)** - highest test F1 score with small train/test gap, confirming strong generalisation through ensemble averaging.